[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/benchmarks/Benchmark_NER.ipynb)

# Benchmark: NER (Named Entity Recognition)

Scores a pretrained NER model's entity-level precision/recall/F1 against gold-labeled data,
using `sparknlp.benchmark.Benchmark.evaluate(..., task="ner")`.

**Dataset**: [WNUT-17](https://noisy-text.github.io/2017/emerging-rare-entities.html) (Emerging
and Rare Entities), dev split -- freely redistributable (unlike CoNLL-2003, which has
redistribution restrictions).

**Model**: the standard `glove_100d` + `ner_dl` combo, Spark NLP's default pretrained English NER
pipeline.

## Setup

Run these cells first on a fresh Colab runtime.

In [3]:
!wget https://setup.johnsnowlabs.com/colab.sh -O - | bash

--2026-08-29 10:57:08--  https://setup.johnsnowlabs.com/colab.sh
Resolving setup.johnsnowlabs.com (setup.johnsnowlabs.com)... 3.86.22.73
Connecting to setup.johnsnowlabs.com (setup.johnsnowlabs.com)|3.86.22.73|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh [following]
--2026-08-29 10:57:08--  https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1483 (1.4K) [text/plain]
Saving to: ‘STDOUT’


-                     0%[                    ]       0  --.-KB/s               
-                   100%[===================>]   1.45K  --.-KB/s    in 0s      



In [4]:
# Current Colab runtimes default to Java 21, which Spark 3.4.x (what the bootstrap above
# installs) isn't compatible with -- Spark's low-level Platform.java reflection breaks on it.
# Switch to Java 17, which Spark 3.4.x does support.
!apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [5]:
# Current Colab runtimes also default to Python 3.13, which removed the deprecated
# `typing.io` submodule -- but Spark 3.4.x's own source still does `from typing.io import
# BinaryIO`. This patches that one import, both for this notebook process and for the
# separate Python worker subprocesses Spark launches to actually run distributed tasks
# (those load pyspark from its own bundled zip, so both copies need patching).
import zipfile, shutil

site_pkgs = "/usr/local/lib/python3.13/dist-packages"
loose_path = f"{site_pkgs}/pyspark/broadcast.py"
zip_path = f"{site_pkgs}/pyspark/python/lib/pyspark.zip"
OLD = "from typing.io import BinaryIO  # type: ignore[import]"
NEW = "from typing import BinaryIO  # patched for Python 3.13 (typing.io removed)"

with open(loose_path) as f:
    text = f.read()
with open(loose_path, "w") as f:
    f.write(text.replace(OLD, NEW))

tmp_path = zip_path + ".tmp"
with zipfile.ZipFile(zip_path, "r") as zin, zipfile.ZipFile(tmp_path, "w", zipfile.ZIP_DEFLATED) as zout:
    for item in zin.infolist():
        data = zin.read(item.filename)
        if item.filename == "pyspark/broadcast.py":
            data = data.decode("utf-8").replace(OLD, NEW).encode("utf-8")
        zout.writestr(item, data)
shutil.move(tmp_path, zip_path)
print("Environment patched for this Colab runtime (Java 17, typing.io).")

Environment patched for this Colab runtime (Java 17, typing.io).

In [6]:
import sparknlp
spark = sparknlp.start()
print("Spark NLP version:", sparknlp.version())
print("Apache Spark version:", spark.version)
from sparknlp.training import CoNLL
from sparknlp.annotator import WordEmbeddingsModel, NerDLModel
from pyspark.ml import Pipeline
from pyspark.sql.functions import expr
import pyspark.sql.functions as F
from sparknlp.benchmark import Benchmark

Spark NLP version: 6.4.2
Apache Spark version: 3.4.4

## 1. Get some data

In [8]:
import urllib.request

url = "https://raw.githubusercontent.com/leondz/emerging_entities_17/master/emerging.dev.conll"
raw = urllib.request.urlopen(url, timeout=30).read().decode("utf-8")

sentences = []
current = []
for line in raw.split("\n"):
    if line.strip() == "":
        if current:
            sentences.append(current)
            current = []
        continue
    token, tag = line.split("\t")
    current.append((token, tag))
if current:
    sentences.append(current)

print(len(sentences), "sentences")
print(sentences[0])

1009 sentences
[('Stabilized', 'O'), ('approach', 'O'), ('or', 'O'), ('not', 'O'), ('?', 'O'), ('That', 'O'), ('´', 'O'), ('s', 'O'), ('insane', 'O'), ('and', 'O'), ('good', 'O'), ('.', 'O')]

`CoNLL().readDataset` parses the classic 4-column CoNLL-2003 layout (space-delimited, tag in
the 4th column) into a DataFrame with `document`, `sentence`, `token`, `pos`, and `label`
columns already built as proper Spark NLP annotations. We build our pipeline on the *existing*
`token` column instead of re-running a `Tokenizer`, so predicted and gold tags stay aligned
token-for-token. WNUT-17 ships as `token<TAB>tag` (2 columns), so we convert to the reader's
expected layout once first.

In [10]:
conll_path = "/tmp/wnut17_dev.conll2003"
with open(conll_path, "w") as f:
    for sent in sentences:
        for token, tag in sent:
            f.write(f"{token} O O {tag}\n")
        f.write("\n")

gold_data = CoNLL().readDataset(spark, conll_path)
gold_data.select("text", "label.result").show(3, truncate=80)
print(gold_data.count(), "sentences loaded")

+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|                                                                            text|                                                                          result|
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|                         Stabilized approach or not ? That ´ s insane and good .|                                            [O, O, O, O, O, O, O, O, O, O, O, O]|
|You should ' ve stayed on Redondo Beach Blvd . you were in the borderlines of...|[O, O, O, O, O, O, B-location, I-location, I-location, O, O, O, O, O, O, O, B...|
|                       All I ' ve been doing is BINGE watching Rick and Morty 😂|[O, O, O, O, O, O, O, O, O, B-creative-work, I-creative-work, I-creative-work...|
+----------------

## 2. Build the pipeline

In [12]:
embeddings = WordEmbeddingsModel.pretrained("glove_100d") \
    .setInputCols(["sentence", "token"]).setOutputCol("embeddings")
ner_model = NerDLModel.pretrained("ner_dl") \
    .setInputCols(["sentence", "token", "embeddings"]).setOutputCol("ner")

pipeline = Pipeline(stages=[embeddings, ner_model])
pipeline_model = pipeline.fit(gold_data)

glove_100d download started this may take some time.
Approximate size to download 145.3 MB

[ | ]
[ / ]
[ — ]
[ \ ]
[OK!]
ner_dl download started this may take some time.
Approximate size to download 13.6 MB

[ | ]
[ / ]
[ — ]
[ \ ]
[OK!]

> **Note: tag vocabularies must match.** `ner_dl` was trained on CoNLL-2003's tag set
> (`PER`/`ORG`/`LOC`/`MISC`), while WNUT-17 uses its own, differently-spelled tags
> (`person`/`location`/`corporation`/`product`/`creative-work`/`group`). `Benchmark.evaluate`
> compares tag strings exactly, on purpose -- fuzzy-matching would hide real errors elsewhere.
> That means scoring raw WNUT-17 tags against `ner_dl`'s CoNLL-style output looks like total
> failure even when every span is right, purely from spelling mismatch. Since there's no
> WNUT-17-trained Spark NLP model to pair with, we map WNUT-17's finer-grained tags onto
> CoNLL-2003's coarser ones below (`product`/`creative-work` -> `MISC`, CoNLL's own catch-all
> category) so the comparison is meaningful. The same issue is why the POS notebook below uses
> `XPOS` instead of `UPOS` -- always check your model and your gold data agree on a tag set
> before trusting a low score.

In [14]:
CONLL_TAG_MAP = {
    "person": "PER", "location": "LOC", "corporation": "ORG", "group": "ORG",
    "product": "MISC", "creative-work": "MISC",
}

def to_conll_tag(tag):
    if tag == "O":
        return "O"
    prefix, entity_type = tag.split("-", 1)
    return f"{prefix}-{CONLL_TAG_MAP.get(entity_type, 'MISC')}"

to_conll_tag_udf = F.udf(lambda tags: [to_conll_tag(t) for t in tags], "array<string>")
gold_data = gold_data.withColumn("gold_tags", to_conll_tag_udf(F.expr("label.result")))

## 3. Run the benchmark

In [16]:
report = Benchmark.evaluate(pipeline_model, gold_data, task="ner", label_col="gold_tags")
print(report)

ner accuracy (n=836): f1=0.3755, precision=0.3663, recall=0.3852
  LOC: f1=0.4681, precision=0.4925, recall=0.4459
  MISC: f1=0.1024, precision=0.1250, recall=0.0868
  ORG: f1=0.1970, precision=0.1361, recall=0.3562
  PER: f1=0.5197, precision=0.5203, recall=0.5191

## Try it yourself

Swap in your own NER model and your own gold-labeled data (in *that* model's own tag
vocabulary) to get a number that means something for your use case.